# Road Accident Data Exploration

Initial exploration of Thai road accident data (2022) for Bangkok and Phuket.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
os.chdir('/home/user/hackathon')
sys.path.insert(0, '/home/user/hackathon')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

from src.data_processing import load_raw_data, parse_datetime, extract_features, filter_city

## 1. Load and Inspect Raw Data

In [2]:
df = load_raw_data('data/raw/accident2022.csv')
print(f'Total rows: {len(df):,}')
print(f'Total columns: {len(df.columns)}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

Loaded 63516 rows with encoding: utf-8-sig
Total rows: 63,516
Total columns: 14

Columns: ['ลำดับ', 'วันที่เกิดเหตุ', 'เวลา', 'จังหวัด', 'สายทาง', 'กม.ที่', 'ประเภทอุบัติเหตุ', 'ความรุนแรง', 'ประเภทรถ', 'จำนวนผู้เสียชีวิต', 'จำนวนผู้บาดเจ็บ', 'LATITUDE', 'LONGITUDE', 'สาเหตุ']


,ลำดับ,วันที่เกิดเหตุ,เวลา,จังหวัด,สายทาง,กม.ที่,ประเภทอุบัติเหตุ,ความรุนแรง,ประเภทรถ,จำนวนผู้เสียชีวิต,จำนวนผู้บาดเจ็บ,LATITUDE,LONGITUDE,สาเหตุ
0,1,2022-01-01,23:42,กรุงเทพมหานคร,จรัญสนิทวงศ์ (Charan Sanitwong),3+661,ชนท้าย (Rear-end collision),บาดเจ็บเล็กน้อย (Minor injury),รถจักรยานยนต์ (Motorcycle),0,1,13.752033,100.522125,หลับใน (Drowsy driving)
1,2,2022-01-01,23:43,กรุงเทพมหานคร,รังสิต-นครนายก (Rangsit-Nakhon Nayok),28+699,ชนท้าย (Rear-end collision),เสียชีวิต (Fatal),รถเก๋ง (Sedan),1,3,13.891936,100.613081,ทัศนวิสัยไม่ดี (Poor visibility)
2,3,2022-01-01,13:46,กรุงเทพมหานคร,แจ้งวัฒนะ (Chaeng Watthana),9+166,ชนข้ามทาง (Cross-traffic collision),ทรัพย์สินเสียหาย (Property damage),รถกระบะ (Pickup truck),0,0,13.842137,100.569480,เมาสุรา (Drunk driving)
3,4,2022-01-01,23:59,กรุงเทพมหานคร,รามคำแหง (Ramkhamhaeng),8+942,ชนประสานงา (Head-on collision),บาดเจ็บเล็กน้อย (Minor injury),รถกระบะ (Pickup truck),0,1,13.792697,100.584502,ฝ่าฝืนสัญญาณไฟ (Running red light)
4,5,2022-01-01,22:35,กรุงเทพมหานคร,พระราม 4 (Rama IV),44+161,ชนประสานงา (Head-on collision),บาดเจ็บเล็กน้อย (Minor injury),รถจักรยานยนต์ (Motorcycle),0,2,13.740199,100.557764,เมาสุรา (Drunk driving)
5,6,2022-01-01,22:13,กรุงเทพมหานคร,จรัญสนิทวงศ์ (Charan Sanitwong),45+337,ชนสิ่งกีดขวาง (Hit fixed object),ทรัพย์สินเสียหาย (Property damage),รถบรรทุก (Truck),0,0,13.781188,100.462451,ทัศนวิสัยไม่ดี (Poor visibility)
6,7,2022-01-01,20:25,กรุงเทพมหานคร,รังสิต-นครนายก (Rangsit-Nakhon Nayok),1+647,เฉี่ยวชน (Sideswipe),บาดเจ็บสาหัส (Serious injury),รถจักรยานยนต์ (Motorcycle),0,1,13.847351,100.600823,ทัศนวิสัยไม่ดี (Poor visibility)
7,8,2022-01-01,20:16,กรุงเทพมหานคร,เจริญกรุง (Charoen Krung),9+839,รถจักรยานยนต์ (Motorcycle accident),ทรัพย์สินเสียหาย (Property damage),รถตู้ (Van),0,0,13.725141,100.529555,ตัดหน้ากระชั้นชิด (Cutting in)
8,9,2022-01-01,10:47,กรุงเทพมหานคร,วิภาวดีรังสิต (Vibhavadi Rangsit),47+461,ชนสิ่งกีดขวาง (Hit fixed object),บาดเจ็บเล็กน้อย (Minor injury),รถเก๋ง (Sedan),0,1,13.824183,100.571502,ฝ่าฝืนสัญญาณไฟ (Running red light)
9,10,2022-01-01,19:25,กรุงเทพมหานคร,พระราม 4 (Rama IV),32+899,ชนข้ามทาง (Cross-traffic collision),บาดเจ็บเล็กน้อย (Minor injury),รถตู้ (Van),0,1,13.737677,100.539893,สภาพถนน (Road condition)


## 2. Missing Values Analysis

In [3]:
key_cols = ['LATITUDE', 'LONGITUDE', 'วันที่เกิดเหตุ', 'เวลา', 'จังหวัด', 'สายทาง']
print('Missing values in key columns:')
for col in key_cols:
    missing = df[col].isna().sum()
    pct = 100 * missing / len(df)
    print(f'  {col}: {missing:,} ({pct:.1f}%)')

print(f'\nOverall missing value summary:')
print(df.isnull().sum().to_string())

Missing values in key columns:
  LATITUDE: 0 (0.0%)
  LONGITUDE: 0 (0.0%)
  วันที่เกิดเหตุ: 0 (0.0%)
  เวลา: 0 (0.0%)
  จังหวัด: 0 (0.0%)
  สายทาง: 0 (0.0%)

Overall missing value summary:
ลำดับ                0
วันที่เกิดเหตุ       0
เวลา                 0
จังหวัด              0
สายทาง               0
กม.ที่               0
ประเภทอุบัติเหตุ     0
ความรุนแรง           0
ประเภทรถ             0
จำนวนผู้เสียชีวิต    0
จำนวนผู้บาดเจ็บ      0
LATITUDE             0
LONGITUDE            0
สาเหตุ               0


## 3. City Breakdown

In [4]:
city_counts = df['จังหวัด'].value_counts()
print('Records by province:')
print(city_counts.to_string())

bkk = df[df['จังหวัด'].str.contains('กรุงเทพ', na=False)]
phuket = df[df['จังหวัด'].str.contains('ภูเก็ต', na=False)]
print(f'\nBangkok records: {len(bkk):,}')
print(f'Phuket records: {len(phuket):,}')

Records by province:
จังหวัด
กรุงเทพมหานคร    57436
ภูเก็ต            6080

Bangkok records: 57,436
Phuket records: 6,080


## 4. Date Range

In [5]:
df = parse_datetime(df)
df = extract_features(df)
print(f'Date range: {df["datetime"].min()} to {df["datetime"].max()}')
print(f'Duration: {(df["datetime"].max() - df["datetime"].min()).days} days')

Date range: 2022-01-01 00:07:00 to 2022-12-31 23:56:00
Duration: 364 days


## 5. Accidents by Hour of Day

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bangkok
bkk_data = df[df['จังหวัด'].str.contains('กรุงเทพ', na=False)]
bkk_hourly = bkk_data['hour'].value_counts().sort_index()
axes[0].bar(bkk_hourly.index, bkk_hourly.values, color='#e74c3c', alpha=0.8)
axes[0].set_title('Bangkok: Accidents by Hour', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Accidents')
axes[0].set_xticks(range(0, 24))

# Phuket
pkt_data = df[df['จังหวัด'].str.contains('ภูเก็ต', na=False)]
pkt_hourly = pkt_data['hour'].value_counts().sort_index()
axes[1].bar(pkt_hourly.index, pkt_hourly.values, color='#3498db', alpha=0.8)
axes[1].set_title('Phuket: Accidents by Hour', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Accidents')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('data/processed/accidents_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()
print('Peak hours: 7-9am (morning rush) and 4-7pm (evening rush)')

Peak hours: 7-9am (morning rush) and 4-7pm (evening rush)


## 6. Accidents by Day of Week

In [7]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (data, city, color) in enumerate([
    (bkk_data, 'Bangkok', '#e74c3c'),
    (pkt_data, 'Phuket', '#3498db')
]):
    daily = data['day_name'].value_counts().reindex(day_order)
    axes[idx].bar(range(7), daily.values, color=color, alpha=0.8)
    axes[idx].set_title(f'{city}: Accidents by Day of Week', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Day of Week')
    axes[idx].set_ylabel('Number of Accidents')
    axes[idx].set_xticks(range(7))
    axes[idx].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.savefig('data/processed/accidents_by_day.png', dpi=150, bbox_inches='tight')
plt.show()
print('Weekend spike visible, especially Friday-Saturday')

Weekend spike visible, especially Friday-Saturday


## 7. Top 10 Most Dangerous Routes

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for idx, (data, city, color) in enumerate([
    (bkk_data, 'Bangkok', '#e74c3c'),
    (pkt_data, 'Phuket', '#3498db')
]):
    top_routes = data['สายทาง'].value_counts().head(10)
    axes[idx].barh(range(len(top_routes)), top_routes.values, color=color, alpha=0.8)
    axes[idx].set_yticks(range(len(top_routes)))
    axes[idx].set_yticklabels(top_routes.index, fontsize=10)
    axes[idx].set_title(f'{city}: Top 10 Dangerous Routes', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Number of Accidents')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.savefig('data/processed/top_dangerous_routes.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3626 (\N{THAI CHARACTER SO SUA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3640 (\N{THAI CHARACTER SARA U}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3586 (\N{THAI CHARACTER KHO KHAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3617 (\N{THAI CHARACTER MO MA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3623 (\N{THAI CHARACTER WO WAEN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3636 (\N{THAI CHARACTER SARA I}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/1237295750.py:15: UserWarning: Glyph 3607 (\N{THAI CHARACTER THO THAHAN}) missing from font(s) DejaVu Sans.


## 8. Accident Severity Distribution

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (data, city) in enumerate([(bkk_data, 'Bangkok'), (pkt_data, 'Phuket')]):
    severity = data['ความรุนแรง'].value_counts()
    colors = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60']
    axes[idx].pie(severity.values, labels=severity.index, colors=colors[:len(severity)],
                  autopct='%1.1f%%', startangle=90)
    axes[idx].set_title(f'{city}: Accident Severity', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('data/processed/severity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3610 (\N{THAI CHARACTER BO BAIMAI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3634 (\N{THAI CHARACTER SARA AA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3604 (\N{THAI CHARACTER DO DEK}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3648 (\N{THAI CHARACTER SARA E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3592 (\N{THAI CHARACTER CHO CHAN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3655 (\N{THAI CHARACTER MAITAIKHU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_5724/145503072.py:10: UserWarning: Glyph 3621 (\N{THAI CHARACTER LO LING}) missing from font(s) DejaVu Sans.
  p

## 9. Monthly Trend

In [10]:
monthly = df.groupby(['month', 'จังหวัด']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
months = range(1, 13)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

if 'กรุงเทพมหานคร' in monthly.columns:
    ax.plot(months, monthly['กรุงเทพมหานคร'].values, 'o-', color='#e74c3c', linewidth=2, markersize=8, label='Bangkok')
if 'ภูเก็ต' in monthly.columns:
    ax2 = ax.twinx()
    ax2.plot(months, monthly['ภูเก็ต'].values, 's-', color='#3498db', linewidth=2, markersize=8, label='Phuket')
    ax2.set_ylabel('Phuket Accidents', color='#3498db')

ax.set_xticks(months)
ax.set_xticklabels(month_names)
ax.set_xlabel('Month')
ax.set_ylabel('Bangkok Accidents', color='#e74c3c')
ax.set_title('Monthly Accident Trend (2022)', fontsize=14, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))

plt.tight_layout()
plt.savefig('data/processed/monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note spikes during January (New Year) and April (Songkran)')

Note spikes during January (New Year) and April (Songkran)


## 10. Summary Statistics

In [11]:
print('=== SUMMARY ===')
print(f'Total accidents: {len(df):,}')
print(f'Bangkok: {len(bkk_data):,} ({100*len(bkk_data)/len(df):.1f}%)')
print(f'Phuket: {len(pkt_data):,} ({100*len(pkt_data)/len(df):.1f}%)')
print(f'\nTotal fatalities: {df["จำนวนผู้เสียชีวิต"].sum():,}')
print(f'Total injuries: {df["จำนวนผู้บาดเจ็บ"].sum():,}')
print(f'\nPeak hour: {df["hour"].mode().values[0]}:00')
print(f'Worst day: {df["day_name"].mode().values[0]}')
print(f'Worst month: month {df["month"].mode().values[0]}')
print(f'\nTop accident type: {df["ประเภทอุบัติเหตุ"].mode().values[0]}')
print(f'Top cause: {df["สาเหตุ"].mode().values[0]}')

=== SUMMARY ===
Total accidents: 63,516
Bangkok: 57,436 (90.4%)
Phuket: 6,080 (9.6%)

Total fatalities: 3,688
Total injuries: 58,880

Peak hour: 17:00
Worst day: Saturday
Worst month: month 4

Top accident type: รถจักรยานยนต์ (Motorcycle accident)
Top cause: หลับใน (Drowsy driving)
